# N123 · 后缀自动机与子串状态

**目标：** 用endpos等价类解释状态、克隆与后缀链接。

**先修：** N121, N122, N104。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** len/link；clone；子串识别；不同子串计数；最长公共连续段。

## 从问题到算法

后缀自动机把具有相同出现结束位置集合的子串合并为状态。一个状态代表一段连续长度，而不是唯一字符串。追加符号时，若原状态长度范围过宽，需要克隆状态拆分；克隆复制转移，但最大长度缩短。后缀链接指向更短的结束位置等价类。

## 最小实现

**本章接口：** `SuffixAutomaton.extend(symbol)`、`count_distinct_substrings_sam(s)`、`longest_common_subarray_sam(a, b)`

In [1]:
class SuffixAutomaton:
    def __init__(self): self.next=[{}]; self.link=[-1]; self.length=[0]; self.last=0
    def extend(self,symbol):
        cur=len(self.next); self.next.append({}); self.length.append(self.length[self.last]+1); self.link.append(0); p=self.last
        while p!=-1 and symbol not in self.next[p]: self.next[p][symbol]=cur; p=self.link[p]
        if p==-1: self.link[cur]=0
        else:
            q=self.next[p][symbol]
            if self.length[p]+1==self.length[q]: self.link[cur]=q
            else:
                clone=len(self.next); self.next.append(self.next[q].copy()); self.length.append(self.length[p]+1); self.link.append(self.link[q])
                while p!=-1 and self.next[p].get(symbol)==q: self.next[p][symbol]=clone; p=self.link[p]
                self.link[q]=self.link[cur]=clone
        self.last=cur; return cur

def count_distinct_substrings_sam(s):
    sam=SuffixAutomaton()
    for c in s: sam.extend(c)
    return sum(sam.length[v]-sam.length[sam.link[v]] for v in range(1,len(sam.next)))

def longest_common_subarray_sam(a,b):
    sam=SuffixAutomaton()
    for x in a: sam.extend(x)
    state=matched=best=0
    for x in b:
        while state and x not in sam.next[state]: state=sam.link[state]; matched=min(matched,sam.length[state])
        if x in sam.next[state]: state=sam.next[state][x]; matched+=1
        else: state=matched=0
        best=max(best,matched)
    return best

## 正确性、前提与复杂度

状态v代表长度在length[link[v]]+1到length[v]之间且endpos相同的子串，因此贡献两长度之差。克隆把旧类划分为应接受较短后缀与保留较长后缀两部分，维持转移所代表的子串集合。扫描第二序列时，失败沿suffix link缩短当前匹配，保留仍可能延伸的最长后缀。

**代价：** 固定字母表下SAM状态与转移数线性，构建与匹配O(n+m)。本Python版本用字典并复制克隆转移，字典操作按平均代价分析；对任意大字母表复制成本需计入。输出非空不同子串数；符号必须可哈希。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

s='ababa'; sam=SuffixAutomaton()
for c in s: sam.extend(c)
show_table(['状态','最短长度','最长长度','后缀链接','转移'],[(v,sam.length[sam.link[v]]+1 if v else 0,sam.length[v],sam.link[v],sam.next[v]) for v in range(len(sam.next))])
print('不同子串',count_distinct_substrings_sam(s))

状态,最短长度,最长长度,后缀链接,转移
0,0,0,-1,"{'a': 1, 'b': 2}"
1,1,1,0,{'b': 2}
2,1,2,0,{'a': 3}
3,2,3,1,{'b': 4}
4,3,4,2,{'a': 5}
5,4,5,3,{}


不同子串 9


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert longest_common_subarray_sam([1,2,3,2,1],[3,2,1,4,7])==3
from itertools import product
words=[''.join(p) for n in range(7) for p in product('ab',repeat=n)]
for s in words:
    sam=SuffixAutomaton()
    for c in s: sam.extend(c)
    assert all(sam.length[sam.link[v]]<sam.length[v] for v in range(1,len(sam.next)))
    brute={s[i:j] for i in range(len(s)) for j in range(i+1,len(s)+1)}
    sa=sorted(range(len(s)),key=lambda i:s[i:]); duplicate=0
    for i,j in zip(sa,sa[1:]):
        h=0
        while i+h<len(s) and j+h<len(s) and s[i+h]==s[j+h]: h+=1
        duplicate+=h
    assert count_distinct_substrings_sam(s)==len(brute)==len(s)*(len(s)+1)//2-duplicate
for a in words[:31]:
    for b in words[:31]:
        want=max((k for i in range(len(a)) for j in range(len(b)) for k in range(1,min(len(a)-i,len(b)-j)+1) if a[i:i+k]==b[j:j+k]),default=0)
        assert longest_common_subarray_sam(a,b)==want
print('N123: 所有本章断言通过')

N123: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 将字符边推广为整数边。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较SAM与平方DP解718，注明并非该题必需算法。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 718. Maximum Length of Repeated Subarray（extension）：[官方入口](https://leetcode.com/problems/maximum-length-of-repeated-subarray/)
- 1044. Longest Duplicate Substring（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。